In [13]:
# ============================================================
# 다방 Selenium 크롤링 v6
# 서울 25개 구별 원/투룸 월세 / 단기월세 매물 수 수집
#
# 핵심:
# - API 사용 X
# - 원/투룸만 수집
# - 월세만 선택
# - 지도 마커의 h1 숫자 + p 지역명을 직접 수집
# - 지도가 동 단위로 확대되어 있으면 자동 줌아웃해서 구 단위 마커 탐색
# ============================================================

# 최초 1회만 설치
# !pip install selenium webdriver-manager pandas openpyxl

import re
import time
import random
from datetime import datetime

import pandas as pd
import numpy as np

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager


# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------
BASE_URL = "https://www.dabangapp.com"

SEOUL_GU_LIST = [
    "종로구", "중구", "용산구", "성동구", "광진구",
    "동대문구", "중랑구", "성북구", "강북구", "도봉구",
    "노원구", "은평구", "서대문구", "마포구", "양천구",
    "강서구", "구로구", "금천구", "영등포구", "동작구",
    "관악구", "서초구", "강남구", "송파구", "강동구"
]

COLLECTED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_RAW_CSV = "dabang_gu_listing_counts_v6.csv"
OUTPUT_SUMMARY_CSV = "dabang_region_summary_v6.csv"
OUTPUT_EXCEL = "dabang_gu_count_result_v6.xlsx"
OUTPUT_MARKER_DEBUG_CSV = "dabang_marker_pairs_debug_v6.csv"


# ------------------------------------------------------------
# 2. 공통 함수
# ------------------------------------------------------------
def random_sleep(a=1.0, b=2.0):
    time.sleep(random.uniform(a, b))


def start_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


def wait_body(driver, timeout=20):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def safe_click(driver, element):
    try:
        element.click()
    except Exception:
        driver.execute_script("arguments[0].click();", element)


def close_popups(driver):
    close_texts = ["닫기", "확인", "오늘 하루 보지 않기", "다시 보지 않기", "취소", "나중에"]

    for txt in close_texts:
        try:
            elems = driver.find_elements(By.XPATH, f"//*[contains(text(), '{txt}')]")
            for elem in elems:
                try:
                    if elem.is_displayed():
                        safe_click(driver, elem)
                        random_sleep(0.3, 0.6)
                except Exception:
                    pass
        except Exception:
            pass


def close_filter_panel(driver):
    try:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
        random_sleep(0.7, 1.2)
    except Exception:
        pass


# ------------------------------------------------------------
# 3. 원/투룸 클릭
# ------------------------------------------------------------
def click_oneroom(driver):
    print("[1] 원/투룸 클릭")

    candidates = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='원/투룸' or contains(normalize-space(text()), '원/투룸')]"
    )

    visible = []

    for elem in candidates:
        try:
            if elem.is_displayed():
                visible.append({
                    "element": elem,
                    "text": elem.text.strip(),
                    "x": elem.location["x"],
                    "y": elem.location["y"]
                })
        except Exception:
            pass

    if not visible:
        raise Exception("원/투룸 버튼을 찾지 못했습니다.")

    target = sorted(visible, key=lambda x: (x["x"], x["y"]))[0]
    print("클릭 대상:", target["text"], target["x"], target["y"])

    safe_click(driver, target["element"])
    random_sleep(2, 3)
    close_popups(driver)


# ------------------------------------------------------------
# 4. 서울특별시 검색
# ------------------------------------------------------------
def search_seoul(driver):
    print("[2] 서울특별시 검색")

    search_inputs = driver.find_elements(
        By.XPATH,
        "//input[contains(@placeholder, '지역') or contains(@placeholder, '지하철') or contains(@placeholder, '대학') or contains(@placeholder, '매물번호')]"
    )

    if not search_inputs:
        print("검색창을 찾지 못했습니다. 현재 지도 기준으로 진행합니다.")
        return False

    search_input = search_inputs[0]
    safe_click(driver, search_input)
    random_sleep(0.5, 1.0)

    try:
        search_input.clear()
    except Exception:
        pass

    search_input.send_keys("서울특별시")
    random_sleep(1.0, 1.5)

    clicked = False
    candidates = driver.find_elements(By.XPATH, "//*[contains(text(), '서울특별시')]")

    for c in candidates:
        try:
            text = c.text.strip()
            if "서울특별시" in text and len(text) < 100 and c.is_displayed():
                safe_click(driver, c)
                clicked = True
                print("자동완성 클릭:", text)
                break
        except Exception:
            pass

    if not clicked:
        search_input.send_keys(Keys.ENTER)
        print("Enter 검색")

    random_sleep(4, 6)
    close_popups(driver)
    return True


# ------------------------------------------------------------
# 5. 거래유형 필터
# ------------------------------------------------------------
def is_deal_panel_open(driver):
    labels = driver.find_elements(By.XPATH, "//*[normalize-space(text())='단기월세만 보기']")
    for label in labels:
        try:
            if label.is_displayed():
                return True
        except Exception:
            pass
    return False


def get_deal_button_text(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "button.dock-btn")

    visible = []

    for btn in buttons:
        try:
            if btn.is_displayed():
                visible.append(btn.text.strip())
        except Exception:
            pass

    return visible[0] if visible else ""


def open_deal_filter(driver):
    if is_deal_panel_open(driver):
        return True

    print("[3] 거래유형 필터 열기")

    buttons = driver.find_elements(By.CSS_SELECTOR, "button.dock-btn")
    visible_buttons = []

    for i, btn in enumerate(buttons):
        try:
            if btn.is_displayed():
                visible_buttons.append({
                    "index": i,
                    "element": btn,
                    "text": btn.text.strip(),
                    "x": btn.location["x"],
                    "y": btn.location["y"]
                })
        except Exception:
            pass

    print("보이는 dock-btn:", [(b["index"], b["text"]) for b in visible_buttons])

    if not visible_buttons:
        raise Exception("거래유형 필터 버튼을 찾지 못했습니다.")

    for b in visible_buttons:
        if any(word in b["text"] for word in ["월세", "전세", "단기월세", "거래유형"]):
            safe_click(driver, b["element"])
            random_sleep(1.2, 2.0)
            return True

    target = sorted(visible_buttons, key=lambda x: (x["y"], x["x"]))[0]
    safe_click(driver, target["element"])
    random_sleep(1.2, 2.0)
    return True


def click_exact_small_text(driver, text):
    elems = driver.find_elements(By.XPATH, f"//*[normalize-space(text())='{text}']")
    visible = []

    for elem in elems:
        try:
            if elem.is_displayed():
                rect = driver.execute_script("""
                    const r = arguments[0].getBoundingClientRect();
                    return {
                        left: r.left,
                        top: r.top,
                        width: r.width,
                        height: r.height
                    };
                """, elem)

                area = rect["width"] * rect["height"]

                if 5 <= rect["width"] <= 200 and 5 <= rect["height"] <= 80:
                    visible.append({
                        "element": elem,
                        "rect": rect,
                        "area": area
                    })
        except Exception:
            pass

    if not visible:
        print(f"'{text}' 요소를 찾지 못했습니다.")
        return False

    target = sorted(visible, key=lambda x: x["area"])[0]["element"]
    safe_click(driver, target)
    random_sleep(1, 1.5)
    return True


def set_monthly_only(driver):
    """
    원래 기본값이 '월세, 전세'이므로 전세만 해제한다.
    월세 버튼은 건드리지 않는다.
    """
    print("[4] 거래유형: 월세만 선택")

    current_text = get_deal_button_text(driver)
    print("현재 거래유형 버튼:", repr(current_text))

    open_deal_filter(driver)
    random_sleep(0.8, 1.2)

    if "전세" in current_text:
        print("전세 선택 상태로 판단 → 전세 클릭해서 해제")
        click_exact_small_text(driver, "전세")
    else:
        print("전세가 이미 해제된 것으로 판단")

    random_sleep(2, 3)
    close_filter_panel(driver)

    print("월세만 선택 처리 완료")


# ------------------------------------------------------------
# 6. 단기월세 체크박스
# ------------------------------------------------------------
def get_short_checkbox_pos(driver):
    labels = driver.find_elements(By.XPATH, "//*[normalize-space(text())='단기월세만 보기']")
    visible = []

    for elem in labels:
        try:
            if elem.is_displayed():
                rect = driver.execute_script("""
                    const r = arguments[0].getBoundingClientRect();
                    return {
                        left: r.left,
                        top: r.top,
                        width: r.width,
                        height: r.height
                    };
                """, elem)

                area = rect["width"] * rect["height"]

                if 5 <= rect["width"] <= 250 and 5 <= rect["height"] <= 80:
                    visible.append({
                        "element": elem,
                        "rect": rect,
                        "area": area
                    })
        except Exception:
            pass

    if not visible:
        return None

    target = sorted(visible, key=lambda x: x["area"])[0]
    rect = target["rect"]

    return {
        "x": rect["left"] - 20,
        "y": rect["top"] + rect["height"] / 2
    }


def get_short_checked_state(driver):
    pos = get_short_checkbox_pos(driver)

    if pos is None:
        return None

    state = driver.execute_script("""
        const x = arguments[0];
        const y = arguments[1];

        const inputs = Array.from(document.querySelectorAll('input'));
        let best = null;
        let bestDist = 999999;

        for (const input of inputs) {
            const r = input.getBoundingClientRect();
            const cx = r.left + r.width / 2;
            const cy = r.top + r.height / 2;
            const dist = Math.abs(cx - x) + Math.abs(cy - y);

            if (dist < bestDist) {
                bestDist = dist;
                best = input;
            }
        }

        if (best && bestDist < 80) {
            return best.checked;
        }

        return null;
    """, pos["x"], pos["y"])

    return state


def click_short_checkbox(driver):
    pos = get_short_checkbox_pos(driver)

    if pos is None:
        print("단기월세만 보기 체크박스 좌표를 찾지 못했습니다.")
        return False

    print("단기월세 체크박스 클릭 좌표:", pos["x"], pos["y"])

    driver.execute_script("""
        const x = arguments[0];
        const y = arguments[1];
        const el = document.elementFromPoint(x, y);
        if (el) el.click();
    """, pos["x"], pos["y"])

    random_sleep(1.5, 2.5)
    return True


def set_short_filter(driver, turn_on=True):
    print(f"[5] 단기월세만 보기: {'ON' if turn_on else 'OFF'}")

    open_deal_filter(driver)
    random_sleep(0.8, 1.2)

    current = get_short_checked_state(driver)
    print("현재 체크 상태:", current)

    if current is not None and current == turn_on:
        close_filter_panel(driver)
        print("이미 원하는 상태입니다.")
        return True

    if current is None and turn_on is False:
        close_filter_panel(driver)
        print("체크 상태를 못 읽었지만 OFF 단계이므로 클릭하지 않고 진행")
        return True

    click_short_checkbox(driver)
    random_sleep(1.5, 2.5)

    after = get_short_checked_state(driver)
    print("변경 후 체크 상태:", after)

    close_filter_panel(driver)
    return True


# ------------------------------------------------------------
# 7. 지도 줌아웃
# ------------------------------------------------------------
def zoom_out_once(driver):
    """
    지도 축소 버튼 클릭 또는 휠 줌아웃.
    """
    close_filter_panel(driver)

    # 1순위: 축소 버튼 찾기
    try:
        candidates = driver.find_elements(
            By.XPATH,
            "//*[contains(@aria-label, '축소') or contains(@title, '축소') or normalize-space(text())='-']"
        )

        for elem in candidates:
            try:
                if elem.is_displayed():
                    safe_click(driver, elem)
                    random_sleep(1.0, 1.5)
                    return True
            except Exception:
                pass
    except Exception:
        pass

    # 2순위: 지도 영역에서 마우스 휠 줌아웃
    try:
        size = driver.get_window_size()
        x = int(size["width"] * 0.65)
        y = int(size["height"] * 0.50)

        origin = ScrollOrigin.from_viewport(x, y)
        ActionChains(driver).scroll_from_origin(origin, 0, 900).perform()
        random_sleep(1.0, 1.5)
        return True
    except Exception:
        pass

    # 3순위: JS wheel 이벤트
    try:
        driver.execute_script("""
            const x = Math.floor(window.innerWidth * 0.65);
            const y = Math.floor(window.innerHeight * 0.50);
            const el = document.elementFromPoint(x, y);

            if (el) {
                const evt = new WheelEvent('wheel', {
                    deltaY: 1200,
                    clientX: x,
                    clientY: y,
                    bubbles: true,
                    cancelable: true
                });
                el.dispatchEvent(evt);
            }
        """)
        random_sleep(1.0, 1.5)
        return True
    except Exception:
        pass

    return False


# ------------------------------------------------------------
# 8. 지도 마커 h1 + p 구조 수집
# ------------------------------------------------------------
def collect_marker_pairs(driver):
    """
    지도 마커 구조에서 h1 숫자 + p 지역명을 직접 읽는다.
    예:
    <h1>102</h1>
    <p>은평구</p>
    """
    markers = driver.execute_script("""
        const results = [];
        const els = Array.from(document.querySelectorAll('div, a, button, span'));

        for (const el of els) {
            const h1 = el.querySelector('h1');
            const p = el.querySelector('p');

            if (!h1 || !p) continue;

            const countText = (h1.innerText || h1.textContent || '').trim();
            const nameText = (p.innerText || p.textContent || '').trim();

            if (!/^\\d{1,5}$/.test(countText)) continue;
            if (!nameText) continue;
            if (nameText.length < 2 || nameText.length > 12) continue;

            const r = el.getBoundingClientRect();

            if (r.width <= 0 || r.height <= 0) continue;
            if (r.bottom < 0 || r.top > window.innerHeight) continue;
            if (r.right < 0 || r.left > window.innerWidth) continue;

            // 왼쪽 사이드바와 필터 패널 제외
            if (r.left < 450) continue;

            // 너무 큰 부모 요소 제외
            if (r.width > 200 || r.height > 100) continue;

            results.push({
                count: parseInt(countText, 10),
                name: nameText,
                left: r.left,
                top: r.top,
                right: r.right,
                bottom: r.bottom,
                width: r.width,
                height: r.height,
                className: String(el.className || ''),
                html: el.outerHTML ? el.outerHTML.slice(0, 300) : ''
            });
        }

        return results;
    """)

    # 중복 제거
    unique = {}
    for m in markers:
        key = (
            m["name"],
            m["count"],
            round(float(m["left"]) / 5),
            round(float(m["top"]) / 5)
        )
        unique[key] = m

    return list(unique.values())


def save_marker_debug(markers, label):
    df = pd.DataFrame(markers)

    if df.empty:
        df.to_csv(OUTPUT_MARKER_DEBUG_CSV, index=False, encoding="utf-8-sig")
    else:
        df["debug_label"] = label
        df.to_csv(OUTPUT_MARKER_DEBUG_CSV, index=False, encoding="utf-8-sig")

    print("마커 디버그 저장:", OUTPUT_MARKER_DEBUG_CSV)

    if not df.empty:
        display(df.head(80))

    return df


def zoom_out_until_gu_markers(driver, target_gu_count=18, max_zoom_out=10):
    """
    현재 화면에서 구 단위 마커가 충분히 나올 때까지 자동 줌아웃.
    """
    print("[6] 구 단위 마커 탐색")

    for attempt in range(max_zoom_out + 1):
        markers = collect_marker_pairs(driver)

        gu_markers = [m for m in markers if m["name"] in SEOUL_GU_LIST]
        dong_markers = [m for m in markers if m["name"].endswith("동")]

        print(
            f"시도 {attempt}: 전체 마커 {len(markers)}개 / 구 마커 {len(gu_markers)}개 / 동 마커 {len(dong_markers)}개"
        )

        if len(gu_markers) >= target_gu_count:
            print("구 단위 마커 확인 완료")
            return gu_markers

        zoom_out_once(driver)
        random_sleep(1.5, 2.5)

    markers = collect_marker_pairs(driver)
    gu_markers = [m for m in markers if m["name"] in SEOUL_GU_LIST]

    print("최대 줌아웃 후 구 마커 수:", len(gu_markers))

    if len(gu_markers) == 0:
        save_marker_debug(markers, "failed_no_gu_markers")
    else:
        save_marker_debug(gu_markers, "partial_gu_markers")

    return gu_markers


# ------------------------------------------------------------
# 9. 구별 매물 수 수집
# ------------------------------------------------------------
def collect_gu_counts_from_map(driver, listing_type, short_month_filter_yn):
    print(f"[7] 지도 구별 매물 수 수집: {listing_type}")

    close_filter_panel(driver)
    random_sleep(2, 3)

    gu_markers = zoom_out_until_gu_markers(
        driver=driver,
        target_gu_count=18,
        max_zoom_out=10
    )

    records = []

    for m in gu_markers:
        if m["name"] not in SEOUL_GU_LIST:
            continue

        records.append({
            "collected_at": COLLECTED_AT,
            "region_sido": "서울특별시",
            "region_sigungu": m["name"],
            "listing_type": listing_type,
            "property_category": "원/투룸",
            "deal_type": "월세",
            "short_month_filter_yn": short_month_filter_yn,
            "listing_count": int(m["count"]),
            "marker_left": m["left"],
            "marker_top": m["top"],
            "marker_class": m["className"],
            "parse_method": "h1_p_marker"
        })

    columns = [
        "collected_at",
        "region_sido",
        "region_sigungu",
        "listing_type",
        "property_category",
        "deal_type",
        "short_month_filter_yn",
        "listing_count",
        "marker_left",
        "marker_top",
        "marker_class",
        "parse_method"
    ]

    df = pd.DataFrame(records)

    if df.empty:
        df = pd.DataFrame(columns=columns)

    # 같은 구가 여러 번 잡히면 큰 값 사용
    if not df.empty:
        df = (
            df.sort_values("listing_count", ascending=False)
            .drop_duplicates(subset=["region_sigungu"])
            .reset_index(drop=True)
        )

    existing = set(df["region_sigungu"].tolist()) if not df.empty else set()

    missing_rows = []
    for gu in SEOUL_GU_LIST:
        if gu not in existing:
            missing_rows.append({
                "collected_at": COLLECTED_AT,
                "region_sido": "서울특별시",
                "region_sigungu": gu,
                "listing_type": listing_type,
                "property_category": "원/투룸",
                "deal_type": "월세",
                "short_month_filter_yn": short_month_filter_yn,
                "listing_count": 0,
                "marker_left": np.nan,
                "marker_top": np.nan,
                "marker_class": None,
                "parse_method": "not_visible_or_not_collected"
            })

    if missing_rows:
        df = pd.concat([df, pd.DataFrame(missing_rows)], ignore_index=True)

    df = df[columns]
    df = df.sort_values("region_sigungu").reset_index(drop=True)

    print("수집된 구 수:", df[df["listing_count"] > 0]["region_sigungu"].nunique())
    display(df[["region_sigungu", "listing_count", "listing_type", "parse_method"]])

    return df


# ------------------------------------------------------------
# 10. 요약 및 저장
# ------------------------------------------------------------
def make_region_summary(df_counts):
    monthly = (
        df_counts[df_counts["short_month_filter_yn"] == False]
        [["region_sido", "region_sigungu", "listing_count"]]
        .rename(columns={"listing_count": "monthly_listing_count"})
    )

    short = (
        df_counts[df_counts["short_month_filter_yn"] == True]
        [["region_sigungu", "listing_count"]]
        .rename(columns={"listing_count": "short_month_listing_count"})
    )

    summary = monthly.merge(short, on="region_sigungu", how="left")

    summary["monthly_listing_count"] = summary["monthly_listing_count"].fillna(0).astype(int)
    summary["short_month_listing_count"] = summary["short_month_listing_count"].fillna(0).astype(int)

    summary["short_month_ratio"] = np.where(
        summary["monthly_listing_count"] > 0,
        (summary["short_month_listing_count"] / summary["monthly_listing_count"] * 100).round(2),
        0
    )

    summary = summary.sort_values("short_month_ratio", ascending=False).reset_index(drop=True)

    return summary


def save_results(df_counts, summary):
    df_counts.to_csv("dabang_gu_listing_counts_v6.csv", index=False, encoding="utf-8-sig")
    summary.to_csv("dabang_region_summary_v6.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter("dabang_gu_count_result_v6.xlsx", engine="openpyxl") as writer:
        df_counts.to_excel(writer, sheet_name="raw_counts", index=False)
        summary.to_excel(writer, sheet_name="region_summary", index=False)

    print("\n저장 완료")
    print("- dabang_gu_listing_counts_v6.csv")
    print("- dabang_region_summary_v6.csv")
    print("- dabang_gu_count_result_v6.xlsx")


# ------------------------------------------------------------
# 11. 최종 실행
# ------------------------------------------------------------
def run_dabang_count_crawling_v6():
    driver = start_driver()

    try:
        print("=" * 80)
        print("다방 서울 25개 구 원/투룸 월세 / 단기월세 매물 수 수집 시작 v6")
        print("=" * 80)

        driver.get(BASE_URL)
        wait_body(driver)
        random_sleep(2, 3)
        close_popups(driver)

        click_oneroom(driver)

        search_seoul(driver)

        set_monthly_only(driver)
        random_sleep(3, 5)

        # 전체 원/투룸 월세
        set_short_filter(driver, turn_on=False)
        random_sleep(3, 5)

        df_monthly = collect_gu_counts_from_map(
            driver=driver,
            listing_type="전체 원/투룸 월세",
            short_month_filter_yn=False
        )

        # 단기월세
        set_short_filter(driver, turn_on=True)
        random_sleep(3, 5)

        df_short = collect_gu_counts_from_map(
            driver=driver,
            listing_type="전체 원/투룸 단기월세",
            short_month_filter_yn=True
        )

        df_counts = pd.concat([df_monthly, df_short], ignore_index=True)
        summary = make_region_summary(df_counts)

        save_results(df_counts, summary)

        print("\n[서울시 25개 구 원/투룸 월세 / 단기월세 요약]")
        display(summary)

        print("\n완료. 브라우저는 확인을 위해 열어둡니다.")

        return {
            "driver": driver,
            "df_counts": df_counts,
            "summary": summary
        }

    except Exception as e:
        print("\n오류 발생:", e)

        try:
            markers = collect_marker_pairs(driver)
            save_marker_debug(markers, "error_debug")
        except Exception:
            pass

        print("브라우저는 확인을 위해 열어둡니다.")

        return {
            "driver": driver,
            "error": e
        }


results_v6 = run_dabang_count_crawling_v6()

다방 서울 25개 구 원/투룸 월세 / 단기월세 매물 수 수집 시작 v6
[1] 원/투룸 클릭
클릭 대상: 원/투룸 22 123
[2] 서울특별시 검색
자동완성 클릭: 서울특별시
[4] 거래유형: 월세만 선택
현재 거래유형 버튼: '월세, 전세'
[3] 거래유형 필터 열기
보이는 dock-btn: [(0, '월세, 전세'), (1, '방크기'), (2, '사용승인일'), (3, '층수'), (4, '추가필터')]
전세 선택 상태로 판단 → 전세 클릭해서 해제
월세만 선택 처리 완료
[5] 단기월세만 보기: OFF
현재 체크 상태: False
이미 원하는 상태입니다.
[7] 지도 구별 매물 수 수집: 전체 원/투룸 월세
[6] 구 단위 마커 탐색
시도 0: 전체 마커 19개 / 구 마커 0개 / 동 마커 15개
시도 1: 전체 마커 66개 / 구 마커 0개 / 동 마커 54개
시도 2: 전체 마커 40개 / 구 마커 20개 / 동 마커 0개
구 단위 마커 확인 완료
수집된 구 수: 20


,region_sigungu,listing_count,listing_type,parse_method
0,강남구,641,전체 원/투룸 월세,h1_p_marker
1,강동구,442,전체 원/투룸 월세,h1_p_marker
2,강북구,0,전체 원/투룸 월세,not_visible_or_not_collected
3,강서구,581,전체 원/투룸 월세,h1_p_marker
4,관악구,1372,전체 원/투룸 월세,h1_p_marker
5,광진구,550,전체 원/투룸 월세,h1_p_marker
6,구로구,346,전체 원/투룸 월세,h1_p_marker
7,금천구,400,전체 원/투룸 월세,h1_p_marker
8,노원구,0,전체 원/투룸 월세,not_visible_or_not_collected
9,도봉구,0,전체 원/투룸 월세,not_visible_or_not_collected


[5] 단기월세만 보기: ON
[3] 거래유형 필터 열기
보이는 dock-btn: [(0, '월세'), (1, '방크기'), (2, '사용승인일'), (3, '층수'), (4, '추가필터')]
현재 체크 상태: False
단기월세 체크박스 클릭 좌표: 126 259.0
변경 후 체크 상태: True
[7] 지도 구별 매물 수 수집: 전체 원/투룸 단기월세
[6] 구 단위 마커 탐색
시도 0: 전체 마커 38개 / 구 마커 20개 / 동 마커 0개
구 단위 마커 확인 완료
수집된 구 수: 20


,region_sigungu,listing_count,listing_type,parse_method
0,강남구,480,전체 원/투룸 단기월세,h1_p_marker
1,강동구,23,전체 원/투룸 단기월세,h1_p_marker
2,강북구,0,전체 원/투룸 단기월세,not_visible_or_not_collected
3,강서구,67,전체 원/투룸 단기월세,h1_p_marker
4,관악구,134,전체 원/투룸 단기월세,h1_p_marker
5,광진구,2,전체 원/투룸 단기월세,h1_p_marker
6,구로구,58,전체 원/투룸 단기월세,h1_p_marker
7,금천구,42,전체 원/투룸 단기월세,h1_p_marker
8,노원구,0,전체 원/투룸 단기월세,not_visible_or_not_collected
9,도봉구,0,전체 원/투룸 단기월세,not_visible_or_not_collected



저장 완료
- dabang_gu_listing_counts_v6.csv
- dabang_region_summary_v6.csv
- dabang_gu_count_result_v6.xlsx

[서울시 25개 구 원/투룸 월세 / 단기월세 요약]


,region_sido,region_sigungu,monthly_listing_count,short_month_listing_count,short_month_ratio
0,서울특별시,강남구,641,480,74.88
1,서울특별시,서초구,159,79,49.69
2,서울특별시,송파구,289,110,38.06
3,서울특별시,구로구,346,58,16.76
4,서울특별시,강서구,581,67,11.53
5,서울특별시,양천구,71,8,11.27
6,서울특별시,금천구,400,42,10.50
7,서울특별시,관악구,1372,134,9.77
8,서울특별시,동작구,586,39,6.66
9,서울특별시,종로구,105,6,5.71



완료. 브라우저는 확인을 위해 열어둡니다.


In [14]:
# ============================================================
# 다방 v6 누락 구 보완 코드
# 대상:
# 강북구, 노원구, 도봉구, 성북구, 은평구
#
# 실행 조건:
# 1. v6 코드가 이미 실행되어 있어야 함
# 2. results_v6["driver"] 브라우저가 열려 있어야 함
# 3. collect_marker_pairs, set_short_filter, search_seoul 등 v6 함수가 메모리에 있어야 함
# ============================================================

import os
import time
import random
import pandas as pd
import numpy as np

from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains


# ------------------------------------------------------------
# 1. 보완 대상 설정
# ------------------------------------------------------------
MISSING_GU_TARGETS = ["강북구", "노원구", "도봉구", "성북구", "은평구"]

SUPPLEMENT_COLLECTED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OUTPUT_COMPLETED_COUNTS = "dabang_gu_listing_counts_v6_completed.csv"
OUTPUT_COMPLETED_SUMMARY = "dabang_region_summary_v6_completed.csv"
OUTPUT_COMPLETED_EXCEL = "dabang_gu_count_result_v6_completed.xlsx"
OUTPUT_SUPPLEMENT_RAW = "dabang_missing_gu_supplement_raw.csv"


# ------------------------------------------------------------
# 2. 드라이버 / 기존 결과 불러오기
# ------------------------------------------------------------
def get_v6_driver():
    if "results_v6" not in globals():
        raise Exception("results_v6 변수가 없습니다. 먼저 v6 코드를 실행해야 합니다.")

    if not isinstance(results_v6, dict) or "driver" not in results_v6:
        raise Exception("results_v6 안에 driver가 없습니다. v6 코드를 다시 실행해야 합니다.")

    driver = results_v6["driver"]

    try:
        print("현재 브라우저 URL:", driver.current_url)
    except Exception:
        raise Exception("브라우저가 닫혀 있습니다. v6 코드를 다시 실행해야 합니다.")

    return driver


def load_v6_counts():
    if "results_v6" in globals() and isinstance(results_v6, dict) and "df_counts" in results_v6:
        print("results_v6['df_counts']에서 기존 결과를 불러옵니다.")
        return results_v6["df_counts"].copy()

    if os.path.exists("dabang_gu_listing_counts_v6.csv"):
        print("dabang_gu_listing_counts_v6.csv에서 기존 결과를 불러옵니다.")
        return pd.read_csv("dabang_gu_listing_counts_v6.csv")

    raise Exception("기존 v6 결과를 찾지 못했습니다. v6 크롤링을 먼저 실행해야 합니다.")


# ------------------------------------------------------------
# 3. 지도 이동 함수
# ------------------------------------------------------------
def supplement_sleep(a=1.2, b=2.2):
    time.sleep(random.uniform(a, b))


def pan_map_by_drag(driver, dx=0, dy=300, label=""):
    """
    지도 영역을 드래그해서 화면을 이동한다.

    dy > 0: 화면상 지도 이미지를 아래로 끌어 북부권을 보이게 하는 용도
    dx > 0: 화면상 지도 이미지를 오른쪽으로 끌어 서북권을 보이게 하는 용도
    dx < 0: 화면상 지도 이미지를 왼쪽으로 끌어 동북권을 보이게 하는 용도
    """
    print(f"[지도 이동] {label} / dx={dx}, dy={dy}")

    try:
        close_filter_panel(driver)
    except Exception:
        pass

    supplement_sleep(0.5, 1.0)

    size = driver.get_window_size()

    # 왼쪽 패널을 피해서 지도 중앙 쪽을 잡는다.
    start_x = int(size["width"] * 0.65)
    start_y = int(size["height"] * 0.50)

    try:
        target = driver.execute_script("""
            const x = arguments[0];
            const y = arguments[1];
            return document.elementFromPoint(x, y);
        """, start_x, start_y)

        ActionChains(driver) \
            .move_to_element(target) \
            .click_and_hold() \
            .move_by_offset(dx, dy) \
            .release() \
            .perform()

        supplement_sleep(2.0, 3.0)
        return True

    except Exception as e:
        print("지도 드래그 실패:", e)
        return False


# ------------------------------------------------------------
# 4. 현재 지도에서 구 마커를 행 데이터로 변환
# ------------------------------------------------------------
def marker_rows_from_current_view(driver, listing_type, short_month_filter_yn, scan_label):
    """
    현재 지도 화면의 h1+p 마커를 읽어서 구 단위 행으로 변환.
    v6의 collect_marker_pairs() 함수를 사용한다.
    """
    markers = collect_marker_pairs(driver)

    gu_markers = [
        m for m in markers
        if m.get("name") in SEOUL_GU_LIST
    ]

    print(
        f"[{scan_label}] 전체 마커 {len(markers)}개 / 구 마커 {len(gu_markers)}개 / "
        f"보완 대상 발견 {[m['name'] for m in gu_markers if m['name'] in MISSING_GU_TARGETS]}"
    )

    rows = []

    for m in gu_markers:
        rows.append({
            "collected_at": SUPPLEMENT_COLLECTED_AT,
            "region_sido": "서울특별시",
            "region_sigungu": m["name"],
            "listing_type": listing_type,
            "property_category": "원/투룸",
            "deal_type": "월세",
            "short_month_filter_yn": short_month_filter_yn,
            "listing_count": int(m["count"]),
            "marker_left": m["left"],
            "marker_top": m["top"],
            "marker_class": m.get("className", ""),
            "parse_method": f"supplement_{scan_label}"
        })

    return rows


# ------------------------------------------------------------
# 5. 한 상태별 보완 수집
# ------------------------------------------------------------
def collect_missing_gu_for_one_state(driver, short_month_filter_yn):
    """
    short_month_filter_yn=False:
        전체 원/투룸 월세 보완 수집

    short_month_filter_yn=True:
        전체 원/투룸 단기월세 보완 수집
    """
    listing_type = "전체 원/투룸 단기월세" if short_month_filter_yn else "전체 원/투룸 월세"

    print("\n" + "=" * 80)
    print(f"보완 수집 시작: {listing_type}")
    print("=" * 80)

    # 서울 화면으로 다시 맞춘 뒤 시작
    search_seoul(driver)
    supplement_sleep(2, 3)

    # 단기월세 ON/OFF 상태 맞추기
    set_short_filter(driver, turn_on=short_month_filter_yn)
    supplement_sleep(2, 3)

    # 구 단위 마커가 나오도록 줌아웃
    zoom_out_until_gu_markers(
        driver=driver,
        target_gu_count=10,
        max_zoom_out=8
    )

    supplement_sleep(1.5, 2.5)

    all_rows = []

    # 1차: 현재 화면
    all_rows.extend(
        marker_rows_from_current_view(
            driver=driver,
            listing_type=listing_type,
            short_month_filter_yn=short_month_filter_yn,
            scan_label="center"
        )
    )

    # 2차: 북부권 보완 이동
    # 순서상 누락된 은평구/성북구/강북구/도봉구/노원구가 보이도록 북쪽, 북서, 북동을 훑는다.
    pan_steps = [
        ("north_center_1", 0, 330),
        ("north_center_2", 0, 260),
        ("north_west", 360, 0),
        ("north_east", -720, 0),
        ("north_east_more", -280, 0),
        ("north_back_center", 420, 0),
    ]

    for label, dx, dy in pan_steps:
        pan_map_by_drag(driver, dx=dx, dy=dy, label=label)

        # 이동 후 구 마커가 너무 적으면 한 번 더 줌아웃
        markers_now = collect_marker_pairs(driver)
        gu_now = [m for m in markers_now if m.get("name") in SEOUL_GU_LIST]

        if len(gu_now) < 5:
            print("구 마커가 적어서 추가 줌아웃 1회")
            zoom_out_once(driver)
            supplement_sleep(1.5, 2.5)

        all_rows.extend(
            marker_rows_from_current_view(
                driver=driver,
                listing_type=listing_type,
                short_month_filter_yn=short_month_filter_yn,
                scan_label=label
            )
        )

        found_targets = sorted({
            row["region_sigungu"]
            for row in all_rows
            if row["region_sigungu"] in MISSING_GU_TARGETS
        })

        print("현재까지 찾은 보완 대상:", found_targets)

        if set(MISSING_GU_TARGETS).issubset(set(found_targets)):
            print("보완 대상 5개 구를 모두 찾았습니다.")
            break

    df = pd.DataFrame(all_rows)

    if df.empty:
        print("보완 수집 결과가 비어 있습니다.")
        return df

    # 같은 구가 여러 번 잡히면 큰 값 우선
    df = (
        df.sort_values("listing_count", ascending=False)
        .drop_duplicates(subset=["region_sigungu", "short_month_filter_yn"])
        .reset_index(drop=True)
    )

    print(f"\n[{listing_type}] 보완 수집 결과")
    display(df[[
        "region_sigungu",
        "listing_count",
        "listing_type",
        "short_month_filter_yn",
        "parse_method"
    ]])

    return df


# ------------------------------------------------------------
# 6. 기존 v6 결과와 보완 결과 병합
# ------------------------------------------------------------
def merge_supplement_with_v6(base_df, supplement_df):
    """
    기존 v6 결과에서 listing_count가 0인 누락 구를
    보완 수집 값으로 업데이트한다.
    """
    completed = base_df.copy()

    # 컬럼 누락 방지
    for col in supplement_df.columns:
        if col not in completed.columns:
            completed[col] = np.nan

    if supplement_df.empty:
        print("보완 데이터가 비어 있어 기존 결과를 그대로 사용합니다.")
        return completed

    update_count = 0

    for _, row in supplement_df.iterrows():
        gu = row["region_sigungu"]
        flag = row["short_month_filter_yn"]
        count = int(row["listing_count"])

        if count <= 0:
            continue

        mask = (
            (completed["region_sigungu"] == gu) &
            (completed["short_month_filter_yn"] == flag)
        )

        if mask.sum() == 0:
            completed = pd.concat([completed, pd.DataFrame([row])], ignore_index=True)
            update_count += 1
            continue

        current_count = int(completed.loc[mask, "listing_count"].fillna(0).iloc[0])

        # 기존값이 0이면 보완값으로 교체
        # 기존값이 있어도 보완값이 더 크면 더 신뢰 가능한 값으로 교체
        if current_count == 0 or count > current_count:
            for col in completed.columns:
                if col in row.index:
                    completed.loc[mask, col] = row[col]
            update_count += 1

    print("업데이트된 행 수:", update_count)

    return completed


def make_completed_summary(df_counts):
    monthly = (
        df_counts[df_counts["short_month_filter_yn"] == False]
        [["region_sido", "region_sigungu", "listing_count"]]
        .rename(columns={"listing_count": "monthly_listing_count"})
    )

    short = (
        df_counts[df_counts["short_month_filter_yn"] == True]
        [["region_sigungu", "listing_count"]]
        .rename(columns={"listing_count": "short_month_listing_count"})
    )

    summary = monthly.merge(short, on="region_sigungu", how="left")

    summary["monthly_listing_count"] = summary["monthly_listing_count"].fillna(0).astype(int)
    summary["short_month_listing_count"] = summary["short_month_listing_count"].fillna(0).astype(int)

    summary["short_month_ratio"] = np.where(
        summary["monthly_listing_count"] > 0,
        (summary["short_month_listing_count"] / summary["monthly_listing_count"] * 100).round(2),
        0
    )

    summary = summary.sort_values("short_month_ratio", ascending=False).reset_index(drop=True)

    return summary


# ------------------------------------------------------------
# 7. 최종 보완 실행
# ------------------------------------------------------------
def run_missing_gu_supplement():
    driver = get_v6_driver()
    base_df = load_v6_counts()

    print("\n기존 v6 결과 중 0으로 잡힌 구")
    display(
        base_df[
            (base_df["region_sigungu"].isin(MISSING_GU_TARGETS)) |
            (base_df["listing_count"] == 0)
        ][[
            "region_sigungu",
            "listing_type",
            "short_month_filter_yn",
            "listing_count",
            "parse_method"
        ]]
    )

    # 전체 월세 보완
    df_monthly_supp = collect_missing_gu_for_one_state(
        driver=driver,
        short_month_filter_yn=False
    )

    # 단기월세 보완
    df_short_supp = collect_missing_gu_for_one_state(
        driver=driver,
        short_month_filter_yn=True
    )

    supplement_df = pd.concat(
        [df_monthly_supp, df_short_supp],
        ignore_index=True
    )

    supplement_df.to_csv(
        OUTPUT_SUPPLEMENT_RAW,
        index=False,
        encoding="utf-8-sig"
    )

    completed_counts = merge_supplement_with_v6(base_df, supplement_df)
    completed_summary = make_completed_summary(completed_counts)

    completed_counts.to_csv(
        OUTPUT_COMPLETED_COUNTS,
        index=False,
        encoding="utf-8-sig"
    )

    completed_summary.to_csv(
        OUTPUT_COMPLETED_SUMMARY,
        index=False,
        encoding="utf-8-sig"
    )

    with pd.ExcelWriter(OUTPUT_COMPLETED_EXCEL, engine="openpyxl") as writer:
        base_df.to_excel(writer, sheet_name="original_v6_counts", index=False)
        supplement_df.to_excel(writer, sheet_name="supplement_raw", index=False)
        completed_counts.to_excel(writer, sheet_name="completed_counts", index=False)
        completed_summary.to_excel(writer, sheet_name="completed_summary", index=False)

    print("\n" + "=" * 80)
    print("누락 구 보완 완료")
    print("=" * 80)

    print("저장 파일:")
    print("-", OUTPUT_SUPPLEMENT_RAW)
    print("-", OUTPUT_COMPLETED_COUNTS)
    print("-", OUTPUT_COMPLETED_SUMMARY)
    print("-", OUTPUT_COMPLETED_EXCEL)

    print("\n[보완 후 서울 25개 구 요약]")
    display(completed_summary)

    print("\n[아직 0으로 남은 구]")
    remaining_zero = completed_summary[
        (completed_summary["monthly_listing_count"] == 0) |
        (completed_summary["short_month_listing_count"] == 0)
    ]

    display(remaining_zero)

    return {
        "driver": driver,
        "base_df": base_df,
        "supplement_df": supplement_df,
        "completed_counts": completed_counts,
        "completed_summary": completed_summary
    }


supplement_results = run_missing_gu_supplement()

현재 브라우저 URL: https://www.dabangapp.com/map/onetwo?sellingTypeList=%5B%22MONTHLY_RENT%22%5D&isShortLease=true&m_lat=37.4684122&m_lng=126.9331093&m_zoom=12
results_v6['df_counts']에서 기존 결과를 불러옵니다.

기존 v6 결과 중 0으로 잡힌 구


,region_sigungu,listing_type,short_month_filter_yn,listing_count,parse_method
2,강북구,전체 원/투룸 월세,False,0,not_visible_or_not_collected
8,노원구,전체 원/투룸 월세,False,0,not_visible_or_not_collected
9,도봉구,전체 원/투룸 월세,False,0,not_visible_or_not_collected
16,성북구,전체 원/투룸 월세,False,0,not_visible_or_not_collected
21,은평구,전체 원/투룸 월세,False,0,not_visible_or_not_collected
27,강북구,전체 원/투룸 단기월세,True,0,not_visible_or_not_collected
33,노원구,전체 원/투룸 단기월세,True,0,not_visible_or_not_collected
34,도봉구,전체 원/투룸 단기월세,True,0,not_visible_or_not_collected
41,성북구,전체 원/투룸 단기월세,True,0,not_visible_or_not_collected
46,은평구,전체 원/투룸 단기월세,True,0,not_visible_or_not_collected



보완 수집 시작: 전체 원/투룸 월세
[2] 서울특별시 검색
자동완성 클릭: 서울특별시
[5] 단기월세만 보기: OFF
[3] 거래유형 필터 열기
보이는 dock-btn: [(0, '단기월세'), (1, '방크기'), (2, '사용승인일'), (3, '층수'), (4, '추가필터')]
현재 체크 상태: True
단기월세 체크박스 클릭 좌표: 126 259.0
변경 후 체크 상태: False
[6] 구 단위 마커 탐색
시도 0: 전체 마커 19개 / 구 마커 0개 / 동 마커 15개
시도 1: 전체 마커 66개 / 구 마커 0개 / 동 마커 54개
시도 2: 전체 마커 40개 / 구 마커 20개 / 동 마커 0개
구 단위 마커 확인 완료
[center] 전체 마커 40개 / 구 마커 20개 / 보완 대상 발견 []
[지도 이동] north_center_1 / dx=0, dy=330
[north_center_1] 전체 마커 40개 / 구 마커 25개 / 보완 대상 발견 ['노원구', '도봉구', '은평구', '성북구', '강북구']
현재까지 찾은 보완 대상: ['강북구', '노원구', '도봉구', '성북구', '은평구']
보완 대상 5개 구를 모두 찾았습니다.

[전체 원/투룸 월세] 보완 수집 결과


,region_sigungu,listing_count,listing_type,short_month_filter_yn,parse_method
0,관악구,1371,전체 원/투룸 월세,False,supplement_center
1,동대문구,811,전체 원/투룸 월세,False,supplement_center
2,중랑구,759,전체 원/투룸 월세,False,supplement_center
3,강남구,641,전체 원/투룸 월세,False,supplement_north_center_1
4,성북구,588,전체 원/투룸 월세,False,supplement_north_center_1
5,동작구,587,전체 원/투룸 월세,False,supplement_north_center_1
6,강서구,581,전체 원/투룸 월세,False,supplement_center
7,강북구,559,전체 원/투룸 월세,False,supplement_north_center_1
8,광진구,550,전체 원/투룸 월세,False,supplement_north_center_1
9,강동구,442,전체 원/투룸 월세,False,supplement_center



보완 수집 시작: 전체 원/투룸 단기월세
[2] 서울특별시 검색
자동완성 클릭: 서울특별시
[5] 단기월세만 보기: ON
[3] 거래유형 필터 열기
보이는 dock-btn: [(0, '월세'), (1, '방크기'), (2, '사용승인일'), (3, '층수'), (4, '추가필터')]
현재 체크 상태: False
단기월세 체크박스 클릭 좌표: 126 259.0
변경 후 체크 상태: True
[6] 구 단위 마커 탐색
시도 0: 전체 마커 15개 / 구 마커 0개 / 동 마커 11개
시도 1: 전체 마커 36개 / 구 마커 0개 / 동 마커 28개
시도 2: 전체 마커 38개 / 구 마커 20개 / 동 마커 0개
구 단위 마커 확인 완료
[center] 전체 마커 38개 / 구 마커 20개 / 보완 대상 발견 []
[지도 이동] north_center_1 / dx=0, dy=330
[north_center_1] 전체 마커 40개 / 구 마커 25개 / 보완 대상 발견 ['노원구', '도봉구', '은평구', '성북구', '강북구']
현재까지 찾은 보완 대상: ['강북구', '노원구', '도봉구', '성북구', '은평구']
보완 대상 5개 구를 모두 찾았습니다.

[전체 원/투룸 단기월세] 보완 수집 결과


,region_sigungu,listing_count,listing_type,short_month_filter_yn,parse_method
0,강남구,480,전체 원/투룸 단기월세,True,supplement_center
1,관악구,134,전체 원/투룸 단기월세,True,supplement_center
2,송파구,110,전체 원/투룸 단기월세,True,supplement_center
3,서초구,79,전체 원/투룸 단기월세,True,supplement_center
4,강서구,67,전체 원/투룸 단기월세,True,supplement_north_center_1
5,구로구,58,전체 원/투룸 단기월세,True,supplement_north_center_1
6,금천구,42,전체 원/투룸 단기월세,True,supplement_center
7,동작구,39,전체 원/투룸 단기월세,True,supplement_north_center_1
8,강동구,23,전체 원/투룸 단기월세,True,supplement_center
9,성북구,18,전체 원/투룸 단기월세,True,supplement_north_center_1


업데이트된 행 수: 11

누락 구 보완 완료
저장 파일:
- dabang_missing_gu_supplement_raw.csv
- dabang_gu_listing_counts_v6_completed.csv
- dabang_region_summary_v6_completed.csv
- dabang_gu_count_result_v6_completed.xlsx

[보완 후 서울 25개 구 요약]


,region_sido,region_sigungu,monthly_listing_count,short_month_listing_count,short_month_ratio
0,서울특별시,강남구,641,480,74.88
1,서울특별시,서초구,159,79,49.69
2,서울특별시,송파구,289,110,38.06
3,서울특별시,구로구,346,58,16.76
4,서울특별시,강서구,581,67,11.53
5,서울특별시,양천구,71,8,11.27
6,서울특별시,금천구,400,42,10.50
7,서울특별시,관악구,1372,134,9.77
8,서울특별시,동작구,587,39,6.64
9,서울특별시,종로구,105,6,5.71



[아직 0으로 남은 구]


,region_sido,region_sigungu,monthly_listing_count,short_month_listing_count,short_month_ratio
